# DFA Synchronization Animation
Finds a reset (synchronizing) word for a DFA, then animates each letter being applied,
showing how the active set of states collapses down to a single state.

In [ ]:
# Cerny 4 synchronization
# Each dot tracks one starting state through the reset word ba^3ba^3b
# Paste this entire cell and run; the animation renders inline.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from collections import Counter
from IPython.display import HTML

# The synchronizing (reset) word for the Cerny 4-state DFA
WORD = "baaabaaab"

# Transition function: 'a' rotates each state forward by 1 (mod 4); 'b' sends state 3->0, all others stay
dst = {"a": lambda i: (i+1)%4, "b": lambda i: 0 if i==3 else i}

# R = radius at which state nodes are placed; NR = radius of each state circle (node size)
R, NR = 1.2, 0.25

# Evenly space 4 states around a circle, starting from the top (offset by -pi/2)
angles = [i*np.pi/2 - np.pi/2 for i in range(4)]

# Map each state index to its 2D (x, y) position on the circle
spos = {i: (R*np.cos(a), R*np.sin(a)) for i, a in enumerate(angles)}

# Simulate all 4 dots through every letter of WORD; dstates[k] = list of states after k letters applied
dstates = [[0,1,2,3]]
for ch in WORD:
    dstates.append([dst[ch](s) for s in dstates[-1]])

# Evaluate a quadratic Bezier curve at parameter t, given control points p0, p1, p2
def bz(p0, p1, p2, t):
    return ((1-t)**2*p0[0]+2*(1-t)*t*p1[0]+t**2*p2[0],
            (1-t)**2*p0[1]+2*(1-t)*t*p1[1]+t**2*p2[1])

# Compute the three Bezier control points for an arc from state i to state j,
# bowing out by distance c (positive = left of direction, negative = right)
def ctrl(i, j, c):
    x0,y0 = spos[i]; x1,y1 = spos[j]
    dx,dy = x1-x0, y1-y0; d = np.hypot(dx,dy); ux,uy = dx/d, dy/d
    s = (x0+ux*NR, y0+uy*NR); e = (x1-ux*NR, y1-uy*NR)
    return s, ((s[0]+e[0])/2-uy*c, (s[1]+e[1])/2+ux*c), e

# Pre-compute Bezier arcs for each 'a' transition (i -> i+1 mod 4), bowing left
a_paths = {i: ctrl(i,(i+1)%4, 0.3) for i in range(4)}

# Pre-compute the single 'b' transition arc (state 3 -> 0), bowing right
b_path = ctrl(3, 0, -0.3)

# Interpolate a dot's position at animation parameter t in [0,1]:
#  self-loops : pulse the dot outward and back along its radial direction
#  normal transitions: follow the appropriate Bezier arc
def move(src, tgt, letter, t):
    if src == tgt:
        x,y = spos[src]; a = np.arctan2(y,x)
        off = np.sin(t*np.pi)*0.25
        return (x+np.cos(a)*off, y+np.sin(a)*off)
    return bz(*(a_paths[src] if letter=="a" else b_path), t)

# Spread multiple dots that land on the same state into a small circle so they remain visible
def jitter(idx, count):
    if count <= 1: return (0,0)
    a = 2*np.pi*idx/count
    return (0.08*np.cos(a), 0.08*np.sin(a))

# static drawing
fig, ax = plt.subplots(figsize=(5.5,5.5), facecolor="#0a0a0a")
ax.set_xlim(-2.1,2.1); ax.set_ylim(-2.4,2.0)
ax.set_aspect("equal"); ax.axis("off")
fig.subplots_adjust(left=.02, right=.98, top=.98, bottom=.02)

# Draw a Bezier arc with a small two-line arrowhead at the endpoint
def draw_edge(p0,p1,p2, col):
    ts = np.linspace(0,1,30)
    ax.plot([(1-t)**2*p0[0]+2*(1-t)*t*p1[0]+t**2*p2[0] for t in ts],
            [(1-t)**2*p0[1]+2*(1-t)*t*p1[1]+t**2*p2[1] for t in ts],
            color=col, lw=1, alpha=.3)
    a = np.arctan2(p2[1]-p1[1], p2[0]-p1[0])
    for s in (-.35,.35):
        ax.plot([p2[0], p2[0]-.08*np.cos(a+s)],
                [p2[1], p2[1]-.08*np.sin(a+s)], color=col, lw=1, alpha=.3)

# Draw all 'a' arcs (blue) and label each one at 40% along the curve
for i in range(4):
    draw_edge(*a_paths[i], "#5588cc")
    lx,ly = bz(*a_paths[i], .4)
    ax.text(lx-.08, ly, "a", color="#5588cc", fontsize=8, alpha=.45,
            ha="center", va="center", fontfamily="monospace")

# Draw the single 'b' arc (orange-red) and its label
draw_edge(*b_path, "#cc6644")
lx,ly = bz(*b_path, .4)
ax.text(lx+.1, ly, "b", color="#cc6644", fontsize=8, alpha=.45,
        ha="center", va="center", fontfamily="monospace")

# Draw faint self-loop circles for states 0-2 
for i in range(3):
    x,y = spos[i]; a = np.arctan2(y,x)
    cx,cy = x+np.cos(a)*(NR+.12), y+np.sin(a)*(NR+.12)
    ax.plot(cx+.08*np.cos(np.linspace(0,2*np.pi,30)),
            cy+.08*np.sin(np.linspace(0,2*np.pi,30)),
            color="#cc6644", lw=.8, alpha=.2)

# Draw the 4 state nodes as dark circles with their index labels
for i in range(4):
    x,y = spos[i]
    ax.add_patch(plt.Circle((x,y), NR, fc="#111", ec="#444", lw=1.5, zorder=3))
    ax.text(x, y, str(i), ha="center", va="center", fontsize=14, color="#555",
            fontfamily="monospace", zorder=4)

# animated dots + label
COLS = ["#ffcc44","#44ddaa","#ff6688","#66aaff"]
dots = [ax.plot([],[],"o",color=c,ms=10,zorder=5,markeredgecolor="#0005",
                markeredgewidth=.5)[0] for c in COLS]
info = ax.text(0,-2.05,"",ha="center",va="center",fontsize=11,
                fontfamily="monospace",color="#888")

# frame timing
# F  = frames spent animating each letter transition
# HS = hold frames at the very start (show initial state set)
# HE = hold frames at the very end (show "synchronized")
# HM = hold frames between transitions (show current active-state set)
F = 15; HS = 10; HE = 14; HM = 6
n = len(WORD)
total = HS + n*(F+HM) + HE  # total frame count for the full animation

def update(frame):
    f = frame
    if f < HS:
        # Hold at start: all four states active
        step, t, label = 0, 0, "{ 0 1 2 3 }"
    elif f >= total-HE:
        # Hold at end: fully synchronized to state 0
        step, t, label = n, 0, "synchronized → 0"
    else:
        f -= HS
        si = min(f//(F+HM), n-1)   # which letter of WORD we are currently on
        rem = f - si*(F+HM)        # frames elapsed within this letter's slot
        if rem < F:
            # Mid-transition: interpolate dots along their Bezier arcs
            step, t = si, rem/F
            label = f"apply '{WORD[si]}'"
        else:
            # Post-transition hold: dots rest at their new states
            step, t = si+1, 0
            label = "{ "+" ".join(map(str,sorted(set(dstates[si+1]))))+" }"

    if t > 0:
        # Animate: move each dot along its Bezier arc
        ch = WORD[step]
        for di in range(4):
            x,y = move(dstates[step][di], dstates[step+1][di], ch, t)
            dots[di].set_data([x],[y])
    else:
        # Static: place dots at their current states, jittering overlapping ones apart
        idx = min(step, n)
        ps = dstates[idx]; counts = Counter(ps); seen = {}
        for di in range(4):
            s = ps[di]; x,y = spos[s]
            seen[s] = seen.get(s,0)
            jx,jy = jitter(seen[s], counts[s])
            seen[s] += 1
            dots[di].set_data([x+jx],[y+jy])
    info.set_text(label)

# Build and display the animation as a looping HTML widget
anim = FuncAnimation(fig, update, frames=total, interval=55, blit=False)
plt.close(fig)
HTML(anim.to_jshtml(default_mode="loop"))